In [1]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, 'patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio.csv'), index_col=0)
        
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        label_index_set = set(label_df.index)
        patch_index_set = set(patch_list)
        and_set = label_index_set & patch_index_set

        patch_list = list(and_set)
        self.label_df = label_df.loc[patch_list]
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, 'patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        label = self.label_df.loc[patch_id].values
        label = torch.Tensor(label)

        return patch_id, data, label

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [2]:
# prepare necessary arguments and WSI sample list

tif_list = '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/'
tif_list = glob(tif_list + '/*[!xlsx|ipynb|ad]')
tif_list.sort()
tif_list

['/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193345',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193346',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193347',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193348',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759311',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759312',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759313',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258463',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258464',
 '/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_L

In [3]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['WSA_LngSP10193345',
 'WSA_LngSP10193346',
 'WSA_LngSP10193347',
 'WSA_LngSP10193348',
 'WSA_LngSP8759311',
 'WSA_LngSP8759312',
 'WSA_LngSP8759313',
 'WSA_LngSP9258463',
 'WSA_LngSP9258464',
 'WSA_LngSP9258467',
 'WSA_LngSP9258468']

In [4]:
# prepare my GPU

gpu_list = [3]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
# prepare the feature extractor model - KimiaNet

model = models.densenet121(pretrained=True)
model.features = nn.Sequential(model.features, nn.AdaptiveAvgPool2d(output_size=(1, 1)))
num_ftrs = model.classifier.in_features
model_final = fully_connected(model.features, num_ftrs, 250+39)

KimiaNetPyTorchWeights_path = "/home/r15user3/Documents/shared_project/Hist2Cell/code/data_preprocessing/KimiaNetPyTorchWeights.pth"
# Checkpoint_path = '/data2/r10user3/Spatial_Gene_Cell_Ratio/code/model_weights/densenet_multi_cell_gene_epoch500_best_cell_average.pth'

checkpoint = torch.load(KimiaNetPyTorchWeights_path)
# checkpoint = torch.load(Checkpoint_path)

new_state_dict = collections.OrderedDict()
for k, v in checkpoint.items():
    if 'fc_4' in k:
        continue
    name = k[7:]  # remove "module."
    new_state_dict[name] = v
model2_dict = model_final.state_dict()
state_dict = {k:v for k,v in new_state_dict.items() if k in model2_dict.keys()}
model2_dict.update(state_dict)
model_final.load_state_dict(model2_dict)
model = model_final
model = model.to(device)
model

/home/r15user3/anaconda3/envs/gene/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/r15user3/anaconda3/envs/gene/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


fully_connected(
  (model): Sequential(
    (0): Sequential(
      (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): ReLU(inplace=True)
      (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (denselayer2): _DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.

In [6]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [7]:
batch_size = 256
patch_dir = "/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location"

graphs = dict()
test_data_list = []

adata = sc.read("/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location/sp.X_norm5e4_log1p.h5ad")
center_col = adata.obs.array_col
center_row = adata.obs.array_row

for slide in test_slides:
    print(slide)
    test_data = PTC_cell(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=8)
    
    spots_coord = pd.read_csv(os.path.join("/home/r15user3/Documents/shared_project/Hist2Cell/data/human_lung_cell2location", slide, "spots.csv"), index_col=0)

    with torch.no_grad():
        model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []
        
        for name, data, label in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            test_label_array.append(label.cpu().detach().numpy())
            
            features, output = model(data)
            test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            
            # test_prediction_array.append(data.cpu().detach().numpy())
            
            # break


    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    for i in range(len(test_label_array)):
        if len(test_label_array[i].shape) <= 1:
            test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_prediction_array = np.concatenate(test_prediction_array)
    test_label_array = np.concatenate(test_label_array)
    test_names = list()
    for names in test_name_array:
        test_names=test_names+names
    test_node_x_y = list()
    for item in test_names:
        test_node_x_y.append([int(center_col[item]), int(center_row[item])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 3 or x2 >= x1 + 3 or y2 <= y1 - 2 or y2 >= y1 + 2:
                    continue
                else:
                    adj[i][j] = 1.0

    graphs[slide] = {
        'features': test_prediction_array,
        'labels': test_label_array,
        'adj': adj,
        'names': test_names,
        'coord': spots_coord.loc[test_names].values,
    }

    print("OK")
    # break

WSA_LngSP10193345
OK
WSA_LngSP10193346
OK
WSA_LngSP10193347
OK
WSA_LngSP10193348
OK
WSA_LngSP8759311
OK
WSA_LngSP8759312
OK
WSA_LngSP8759313
OK
WSA_LngSP9258463
OK
WSA_LngSP9258464
OK
WSA_LngSP9258467
OK
WSA_LngSP9258468
OK


In [8]:
graphs['WSA_LngSP10193345']['features'].shape

(3234, 1024)

In [9]:
graphs['WSA_LngSP10193345']['coord'].shape

(3234, 2)

In [10]:
graphs['WSA_LngSP10193345']['names']

['WSA_LngSP10193345_GAATGCGAATCGGTTC-1',
 'WSA_LngSP10193345_CTGTGCAGGGTAGGTC-1',
 'WSA_LngSP10193345_CCAGAAAGCAACTCAT-1',
 'WSA_LngSP10193345_CTGGTTTCGAGCAAGA-1',
 'WSA_LngSP10193345_TACCCTCGGTAACCCT-1',
 'WSA_LngSP10193345_TAGCGTTGGGTCTTAC-1',
 'WSA_LngSP10193345_TGACAACTTAAAGGTG-1',
 'WSA_LngSP10193345_TTGCGGAAAGCTGCCC-1',
 'WSA_LngSP10193345_ACTTATTAGGATCGGT-1',
 'WSA_LngSP10193345_CTCTTCTGGAAGTTAG-1',
 'WSA_LngSP10193345_GGAAACCTTGTTGAAT-1',
 'WSA_LngSP10193345_AACTGAGGTCAGCGTC-1',
 'WSA_LngSP10193345_GGGCGGGTTCCCTACG-1',
 'WSA_LngSP10193345_ACGCATAAATGACATG-1',
 'WSA_LngSP10193345_ACACTGATCAAGGTGT-1',
 'WSA_LngSP10193345_CAGTCGAGGATGCAAT-1',
 'WSA_LngSP10193345_AGCCCGCACTACAATG-1',
 'WSA_LngSP10193345_CAATACGAGAGTCTGA-1',
 'WSA_LngSP10193345_GTTCAGTCGCCAAATG-1',
 'WSA_LngSP10193345_GCCTATTTGCTACACA-1',
 'WSA_LngSP10193345_AATGTGCTAATCTGAG-1',
 'WSA_LngSP10193345_CCCTATGTAGAGCAGA-1',
 'WSA_LngSP10193345_CGTGTCTCGTTACGAC-1',
 'WSA_LngSP10193345_AATTCTAGAGTTAGGC-1',
 'WSA_LngSP10193

In [11]:
graph = graphs
for slide in graph:
    print(slide)

WSA_LngSP10193345
WSA_LngSP10193346
WSA_LngSP10193347
WSA_LngSP10193348
WSA_LngSP8759311
WSA_LngSP8759312
WSA_LngSP8759313
WSA_LngSP9258463
WSA_LngSP9258464
WSA_LngSP9258467
WSA_LngSP9258468


In [12]:
# save the graphs

save_path='/home/r15user3/Documents/shared_project/Hist2Cell/code/data_preprocessing/extracted_graph/human_lung_cell2location_high250gene_celltype80_KimiaNet_extracted_graphs'
joblib.dump(graphs, save_path)

['/home/r15user3/Documents/shared_project/Hist2Cell/code/data_preprocessing/extracted_graph/human_lung_cell2location_high250gene_celltype80_KimiaNet_extracted_graphs']